# GTSRB Road Traffic Sign Recognition: Exploratory Data Analysis & Deep Vision Pipeline

**Project ID**: DL-04  
**Domain**: Part B: Deep Learning (Computer Vision & Autonomous Systems)  
**Author**: Bleep AI Engineering Team  

### Abstract
This notebook conducts exploratory analysis on the 43-class German Traffic Sign Recognition Benchmark (GTSRB). We investigate the impact of outdoor illumination variation, implement Contrast-Limited Adaptive Histogram Equalization (CLAHE) in the CIELAB color space, and explore the multi-scale convolutional architecture inspired by Pierre Sermanet and Yann LeCun (2011).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Ensure project src is in path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import GTSRB_CLASSES, GTSRB_CATEGORIES, generate_synthetic_gtsrb_data, get_data_loaders
from src.preprocessor import apply_clahe_lab
from src.model import TrafficSignMultiScaleCNN
from src.evaluate import analyze_speed_vs_warning_confusion

print(f"PyTorch Version: {torch.__version__}")
print(f"Total GTSRB Classes: {len(GTSRB_CLASSES)}")

## 1. Class Taxonomy & Functional Grouping
The 43 classes of GTSRB represent distinct regulatory directives. For robust ADAS design, we organize them into 5 functional categories.

In [ ]:
for category, class_ids in GTSRB_CATEGORIES.items():
    print(f"Category: {category} ({len(class_ids)} classes)")
    sample_names = [GTSRB_CLASSES[cid] for cid in class_ids[:3]]
    print(f"  Examples: {', '.join(sample_names)}...\n")

## 2. Dataset Synthesis & Inspection
In zero-socket environments, our deterministic generator produces authentic geometric traffic sign patterns.

In [ ]:
data_dir = Path("../data/raw")
samples = generate_synthetic_gtsrb_data(data_dir, n_samples_per_class=2, seed=42)
print(f"Generated {len(samples)} synthetic sign images across 43 classes.")

## 3. Preprocessing: CLAHE in CIELAB Color Space
Outdoor signs undergo wide variations in contrast. Applying CLAHE solely to the Luminance (L) channel equalizes local shadows without altering color ratios.

In [ ]:
sample_path = samples[2][0]  # Speed limit 50
orig_img = Image.open(sample_path)
orig_np = np.array(orig_img)
clahe_np = apply_clahe_lab(orig_np, clip_limit=2.0, tile_grid_size=(8, 8))

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(orig_np)
axes[0].set_title("Original Input")
axes[0].axis("off")

axes[1].imshow(clahe_np)
axes[1].set_title("LAB CLAHE Enhanced")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Multi-Scale Convolutional Architecture
We instantiate `TrafficSignMultiScaleCNN` and inspect the intermediate skip connection dimensionality.

In [ ]:
model = TrafficSignMultiScaleCNN(num_classes=43)
dummy_input = torch.randn(2, 3, 32, 32)
stage2_f, stage3_f, fused = model.extract_features(dummy_input)

print(f"Stage 2 Feature Map Shape: {stage2_f.shape}")
print(f"Stage 3 Feature Map Shape: {stage3_f.shape}")
print(f"Fused Representation Dimension: {fused.shape}")
print(f"Total Trainable Parameters: {model.count_parameters():,}")

## 5. Speed Sign vs Warning Sign Confusion Diagnostics

In [ ]:
# Simulated realistic predictions to demonstrate diagnostic error isolation
y_true = [1, 2, 3, 18, 19, 25, 33, 14]
y_pred = [1, 3, 3, 18, 19, 25, 33, 14]  # 50 km/h misclassified as 60 km/h

diag = analyze_speed_vs_warning_confusion(y_true, y_pred)
print("Diagnostic Confusion Analysis:")
for k, v in diag.items():
    print(f"  {k}: {v}")